In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.managed.is_last_step import RemainingSteps
import operator

class QAState(TypedDict):
    answer: str
    history: Annotated[list, operator.add]
    remaining_steps: RemainingSteps

In [2]:
def answer_question(state: QAState):
    print(f"Answering... History: {state['history']}")
    return {"answer": "not good", "history": ["tried"]}

def fix_answer(state: QAState):
    print(f"Fixing... Previous answer: {state['answer']}")
    return {"answer": "still not good", "history": ["fixed"]}

In [3]:
from typing import Literal
from langgraph.graph import END


def should_retry(state: QAState) -> Literal["fix", END]:
    if state["remaining_steps"] <= 1:
        return END
    return "fix"

In [4]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(QAState)

builder.add_node("answer", answer_question)
builder.add_node("fix", fix_answer)

builder.set_entry_point("answer")

builder.add_conditional_edges("answer", should_retry)
builder.add_conditional_edges("fix", should_retry)
builder.add_edge("fix", END)

graph = builder.compile()

In [6]:
from langgraph.errors import GraphRecursionError


result = graph.invoke({"answer": "", "history": []}, {"recursion_limit": 4})
print(result)


Answering... History: []
Fixing... Previous answer: not good
Fixing... Previous answer: still not good
{'answer': 'still not good', 'history': ['tried', 'fixed', 'fixed']}


In [7]:
# This is risky
graph.invoke({"answer": "", "history": []})

Answering... History: []
Fixing... Previous answer: not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good
Fixing... Previous answer: still not good


{'answer': 'still not good',
 'history': ['tried',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed']}

In [8]:
len(['tried',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed',
  'fixed'])

24